# 06 — Forecast Evaluation and Diagnostics

Inspect chronological holdout accuracy, compare against the naive forecast,
and examine residual bias and error concentration.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
import json
import plotly.express as px
import plotly.graph_objects as go

metrics_path = OUTPUT_DIR / "models" / "evaluation_metrics.json"
predictions_path = OUTPUT_DIR / "exports" / "model_test_predictions.csv"
evaluation = json.loads(metrics_path.read_text(encoding="utf-8"))
predictions = pd.read_csv(predictions_path, parse_dates=["Date"])
print(json.dumps(evaluation, indent=2))
predictions.describe().round(2)

In [ ]:
predictions["Absolute Error"] = predictions["Ridge Residual"].abs()
predictions["Absolute Percentage Error"] = (
    predictions["Absolute Error"]
    / predictions["Actual Total System Load T+7"].replace(0, np.nan)
    * 100
)
print({
    "mean_residual": round(float(predictions["Ridge Residual"].mean()), 3),
    "median_absolute_error": round(float(predictions["Absolute Error"].median()), 3),
    "p90_absolute_error": round(float(predictions["Absolute Error"].quantile(0.90)), 3),
    "maximum_absolute_error": round(float(predictions["Absolute Error"].max()), 3),
})

In [ ]:
comparison = go.Figure()
comparison.add_trace(go.Scatter(
    x=predictions["Date"], y=predictions["Actual Total System Load T+7"],
    name="Actual", line={"color": "#163B65", "width": 2.5},
))
comparison.add_trace(go.Scatter(
    x=predictions["Date"], y=predictions["Ridge Prediction"],
    name="Ridge", line={"color": "#2E75B6", "width": 2},
))
comparison.add_trace(go.Scatter(
    x=predictions["Date"], y=predictions["Naive Current Load Prediction"],
    name="Naive", line={"color": "#66788A", "dash": "dot"},
))
comparison.update_layout(
    title="Seven-Day Total System Load Forecast — Chronological Holdout",
    template="plotly_white", hovermode="x unified",
    xaxis_title="Forecast origin date", yaxis_title="Children under care",
)
comparison

In [ ]:
residual_chart = px.histogram(
    predictions,
    x="Ridge Residual",
    nbins=30,
    title="Ridge Forecast Residual Distribution",
    color_discrete_sequence=["#167C80"],
)
residual_chart.update_layout(template="plotly_white", yaxis_title="Observations")
residual_chart

## Deployment gate

Do not deploy solely because a single holdout score improves on the naive
baseline. Require rolling-origin stability, residual review during surge
periods, documented monitoring thresholds, and reproducible retraining.